# Storage layout, fast reads, publishing and packs

This notebook is about how array fields sit on disk and how to read them fast when there are
many records or many small reads:

- **Layouts**: chunk and shard shapes per field or per record type, outside the key.
- **One open per record**: consolidated metadata and a cached zarr group per handle.
- **`record.read`**: many row ranges and index gathers of several fields in one call.
- **`publish`**: copying finished records to write-once storage such as an S3 mount.
- **Packs**: many small records of one type stored as one record.

In [ ]:
import json
import tempfile
from pathlib import Path

import numpy as np

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-storage-"))
td.set_root(root / "local")

## Layouts

An array field is a zarr v3 array, cut into **chunks** (the unit zarr reads and decompresses)
and optionally grouped into **shards** (one file holding many chunks plus an index of their byte
ranges). Small chunks read fewer unneeded bytes; shards keep the number of files small.

`td.Layout` gives target sizes in uncompressed bytes (`chunk_bytes`, `shard_bytes`) or explicit
shapes (`chunks`, `shards`). Chunks run along axis 0 and span the other axes whole, which suits
reading row ranges. A record type sets layouts with the class attribute `layouts`, by field name
or `"*"` for every array field; `write(..., layout=...)` overrides both.

In [ ]:
class Mesh(td.Record):
    layouts = {
        "*": td.Layout(chunk_bytes=32 << 10, shard_bytes=1 << 20),
        "faces": td.Layout(chunk_bytes=128 << 10),  # chunks only, one file per chunk
    }

    def __init__(self, /, *, n_vertices: int, seed: int = 0):
        rng = np.random.default_rng(seed)
        self.vertices = rng.normal(size=(n_vertices, 3)).astype("f4")
        self.normals = rng.normal(size=(n_vertices, 3)).astype("f2")
        self.faces = rng.integers(0, n_vertices, size=(2 * n_vertices, 3)).astype("i4")
        self.write("labels", rng.integers(0, 10, n_vertices).astype("u1"), layout=td.Layout(chunk_bytes=4 << 10))


mesh = Mesh(n_vertices=200_000)
for name, info in mesh.meta["fields"].items():
    print(f"{name:<9} {info['dtype']:<4} {str(info['shape']):<12} {info['layout']}")

`vertices` (12 bytes per row) gets chunks of 2,730 rows, and its 2.4 MB are split into shards of
1 MB. The default for `write` is 128 KB chunks in shards of up to 128 MB: an array that fits in
one chunk stays one chunk, and an array up to 128 MB is one file. Count the files:

In [ ]:
store = mesh.path / "data.zarr"
for name in mesh.keys():
    files = [p for p in (store / name).rglob("*") if p.is_file()]
    print(f"{name:<9} {len(files):>4} files")

The layout never enters the key: the same settings with another layout give the same record.
A layout can still vary per record if the build takes it as a setting with `hashed=False` and
passes it to `write`.

In [ ]:
class Tuned(td.Record):
    def __init__(self, /, *, n: int, layout: td.Layout | None = td.field(None, hashed=False)):
        self.write("x", np.arange(n), layout=layout)


small = Tuned(n=100_000, layout=td.Layout(chunk_bytes=8 << 10))
print(small.key == Tuned.compute_key(n=100_000), small.meta["fields"]["x"]["layout"])

Appended fields (`append`) default to 128 KB chunks without shards, because every append
rewrites the last shard. A field keeps the layout it was created with.

## One open per record

When a build finishes, the metadata of every zarr array goes into the root
`data.zarr/zarr.json` (zarr's consolidated metadata). A record handle opens the zarr group once
and keeps its arrays, so `record[name]` costs no metadata read after the first. Handles of
finished records also keep each shard's index after its first read; zarr would otherwise read it
again before every read from that shard. Workspace records rewrite the consolidated metadata on
every edit, and their handles reopen the group when `_meta.json` changes.

In [ ]:
consolidated = json.loads((store / "zarr.json").read_text())["consolidated_metadata"]["metadata"]
print(sorted(consolidated))

## Many small reads in one call: `record.read`

A typical lookup reads a few dozen row ranges from some fields, then gathers rows of other fields
by index. Reading them one slice at a time pays one request per slice, which on object storage
means one round trip each. `record.read({name: rows, ...})` takes all of one step at once:

- `rows` is a slice, a list of slices or `(start, stop)` pairs (rows concatenated in request
  order), or an integer or boolean index array (rows gathered in the given order).
- Ranges less than a chunk apart are merged into one read, and all reads of the call run
  concurrently.

In [ ]:
rng = np.random.default_rng(1)
starts = rng.integers(0, 199_000, 30)
ranges = [(int(a), int(a) + 40) for a in starts]
faces = mesh.read({"faces": ranges})["faces"]                 # step 1: faces near some places
corners = np.unique(faces)                                   # step 2: their vertices, by index
rows = mesh.read({"vertices": corners, "normals": corners, "labels": corners})
print(faces.shape, {k: v.shape for k, v in rows.items()})
np.testing.assert_array_equal(rows["vertices"], mesh.vertices[corners])

With the `tensorstore` extra installed, `engine="tensorstore"` (or
`$TENSORDIR_READ_ENGINE=tensorstore`) reads the same zarr files with tensorstore, which issued
fewer, larger reads and was about twice as fast as zarr-python on local disk in tensordir's
benchmark. Fields it cannot open fall back to zarr.

In [ ]:
import importlib.util

if importlib.util.find_spec("tensorstore"):
    fast = mesh.read({"vertices": corners}, engine="tensorstore")["vertices"]
    np.testing.assert_array_equal(fast, rows["vertices"])
    print("tensorstore read", fast.shape)

## Publishing to write-once storage

Object-store mounts (mountpoint-s3 for S3) take each file as one sequential write, with no
rename or overwrite. Builds need both: they write into a temporary directory that is renamed into
place, and their leases rewrite a heartbeat file. So build in a local root, then publish:
`record.publish(root)` streams each file once, data first and `_meta.json` last, so a reader of
the destination sees `absent` until the copy is complete.

In [ ]:
shared = root / "shared"   # stands in for an S3 mount
copy = mesh.publish(shared)
print(copy, copy.path.relative_to(root))
print(Mesh.lookup(n_vertices=200_000, root=shared) == copy)   # found by its key, nothing built

A record already in the destination with the same key is kept; a partial copy is removed and
redone. `upstream=True` also publishes the records this one was built from, so its inputs reopen
in the destination. A root created by `publish` is marked read-only: builds and workspace edits
there raise `ReadOnlyError`. From the command line, `tensordir --root ROOT publish DEST [TYPE]`
publishes every done record.

In [ ]:
try:
    Mesh(n_vertices=10, root=shared)
except td.ReadOnlyError as exc:
    print("ReadOnlyError:", exc)

## Packs of small records

Every record costs a directory, `_meta.json`, a zarr store and at least two reads to open. For
millions of small records that are read together, that overhead dominates. A **pack** stores
many records of one type as one record:

- `T.call(*inputs, **settings)` binds and keys a call without building or touching the disk.
- `td.Pack(T)(calls)` builds every member once into one record (or reopens it). Each array field
  of the members becomes one pack array, rows concatenated in key order, with row offsets in
  `<name>.offsets`; JSON fields, specs and inputs go into the `members` field.
- `pack.member(call)` is a read-only `T` with the key the standalone record would have.

In [ ]:
class Segment(td.Record):
    def __init__(self, segment_id: int, /, *, resolution: float = 1.0):
        rng = np.random.default_rng(segment_id)
        n = int(rng.integers(5, 50))
        self.points = (rng.normal(size=(n, 3)) * resolution).astype("f4")
        self.volume = float(n * resolution**3)


calls = [Segment.call(i, resolution=0.5) for i in range(200)]   # nothing is built yet
pack = td.Pack(Segment)(calls)
print(pack, pack.keys())
print(sum(1 for p in pack.path.rglob("*") if p.is_file()), "files for", len(pack.member_keys()), "segments")

In [ ]:
member = pack.member(calls[7])
print(member, isinstance(member, Segment))
print(member.key == Segment.compute_key(7, resolution=0.5), member.spec, member.load("volume"))
print(member.points.shape, member.read({"points": [0, -1]})["points"])

The pack's key is the digest of its sorted member keys, so the same calls in any order reopen
the same pack, and a different grouping is a different pack. A member that already exists as a
finished standalone record is read instead of rebuilt. `pack.members` is a `Collection` of all
members, with the same table as standalone records:

In [ ]:
print(td.Pack(Segment)(calls[::-1]) == pack)
pack.members.table(sizes=False).head()

Members are read-only: writes raise `ReadOnlyError`, and a pack is published as a whole. Members
hold numpy arrays (stored as zarr arrays in the pack) and JSON values; a field's trailing shape and
dtype must agree across members. `Segment.get(...)` and `Segment.lookup(...)` find standalone
records only; a packed member is found through its pack.